# Colab GPU Training for RESD

Notebook flow:
1. mount Google Drive for checkpoints,
2. install missing dependencies,
3. optionally set `HF_TOKEN` and `WANDB_API_KEY`,
4. clone the repo from GitHub,
5. warm up the RESD dataset cache from Hugging Face,
6. run training and stream metrics to Weights & Biases.

Before running:
- in Colab select `Runtime -> Change runtime type -> GPU`,
- update `REPO_URL` below if you train from your own fork,
- if you want persistent checkpoints, keep the Drive mount cell enabled.


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "hydra": "hydra-core",
    "omegaconf": "omegaconf",
    "datasets": "datasets",
    "huggingface_hub": "huggingface_hub",
    "sklearn": "scikit-learn",
    "torchaudio": "torchaudio",
    "soundfile": "soundfile",
    "wandb": "wandb",
}

missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Installed:" if missing else "Already available:", missing)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/aibryanov/speech-emo-recognition.git"
REPO_REF = "master"
HF_DATASET_NAME = "Aniemore/resd"

REPO_DIR = Path("/content/speech-emo-recognition")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/ser_checkpoints")

EXPERIMENT_NAME = "resd_colab_baseline"
WANDB_ENABLED = True
WANDB_PROJECT = "speech-emo-recognition"
WANDB_ENTITY = None

CLI_OVERRIDES = [
    "dataset=resd",
    "experiment=resd",
    "train.device=cuda",
    f"experiment_name={EXPERIMENT_NAME}",
    f"train.checkpoint.dir={CHECKPOINT_DIR.as_posix()}",
    f"wandb.enabled={str(WANDB_ENABLED).lower()}",
    f"wandb.project={WANDB_PROJECT}",
    f"wandb.run_name={EXPERIMENT_NAME}",
    "wandb.tags=[colab,resd]",
    "wandb.watch_model=true",
]

if WANDB_ENTITY:
    CLI_OVERRIDES.append(f"wandb.entity={WANDB_ENTITY}")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    print("$", " ".join(cmd), flush=True)
    process = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)

print("Repo:", REPO_URL)
print("Checkpoints:", CHECKPOINT_DIR)
print("Overrides:")
print(*CLI_OVERRIDES, sep="\n")


In [ ]:
from getpass import getpass
import os
import wandb

if not os.getenv("HF_TOKEN"):
    token = getpass("HF token (optional, press Enter to skip): ")
    if token:
        os.environ["HF_TOKEN"] = token

if WANDB_ENABLED and not os.getenv("WANDB_API_KEY"):
    wandb_key = getpass("W&B API key (optional, press Enter to skip): ")
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key

token = os.getenv("HF_TOKEN")
print("HF_TOKEN set:" , bool(token))
if token:
    print(token[:6] + "..." + token[-4:])

wandb_key = os.getenv("WANDB_API_KEY")
print("WANDB_API_KEY set:", bool(wandb_key))
if WANDB_ENABLED and wandb_key:
    wandb.login(key=wandb_key, relogin=True)


In [ ]:
run(["nvidia-smi"])


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)])
print(f"Repo cloned to: {REPO_DIR}")


In [ ]:
from datasets import load_dataset
import os

dataset = load_dataset(HF_DATASET_NAME, token=os.getenv("HF_TOKEN") or None)
print(dataset)


In [ ]:
import os
import sys
import torch.nn as nn
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.loader import create_dataloaders
from src.model import LSTMClassifier
from src.train import build_classification_details, evaluate, print_classification_details, train, _get_dataset_class_names
from src.wandb_utils import finish_wandb, init_wandb, log_wandb_confusion_matrix, log_wandb_metrics

GlobalHydra.instance().clear()
with initialize_config_dir(version_base="1.3", config_dir=str(REPO_DIR / "configs")):
    cfg = compose(config_name="config", overrides=CLI_OVERRIDES)

if cfg.wandb.enabled and not os.getenv("WANDB_API_KEY"):
    print("WANDB_API_KEY is not set, disabling W&B logging for this run.", flush=True)
    cfg.wandb.enabled = False

print("Config:", flush=True)
print(OmegaConf.to_yaml(cfg), flush=True)
print("Creating dataloaders...", flush=True)
train_loader, dev_loader, test_loader = create_dataloaders(cfg)
print(
    f"Dataloaders ready: train={len(train_loader.dataset)}, dev={len(dev_loader.dataset)}, test={len(test_loader.dataset)}",
    flush=True,
)

print("Building model...", flush=True)
model = LSTMClassifier(cfg)
wandb_run = init_wandb(cfg, model)
if wandb_run is not None:
    print(f"W&B run: {wandb_run.url}", flush=True)

history = None
try:
    print("Starting training...", flush=True)
    history = train(cfg, model, train_loader, dev_loader, wandb_run=wandb_run)

    print("Running test evaluation...", flush=True)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=cfg.train.get("label_smoothing", 0.0))
    test_metrics, y_true, y_pred = evaluate(
        model,
        test_loader,
        loss_fn,
        cfg.train.device,
        average=cfg.train.get("metrics_average", "macro"),
        return_predictions=True,
    )

    log_wandb_metrics(wandb_run, test_metrics, split="test", epoch=cfg.train.epochs)
    class_names = _get_dataset_class_names(test_loader.dataset)
    details = build_classification_details(y_true, y_pred, class_names)
    print_classification_details(details, split="TEST")
    log_wandb_confusion_matrix(wandb_run, y_true, y_pred, class_names, split="test")
    print("TEST METRICS", flush=True)
    print(test_metrics, flush=True)
finally:
    finish_wandb(wandb_run)

history


In [ ]:
outputs_dir = REPO_DIR / "outputs"

if outputs_dir.exists():
    print("Hydra outputs:")
    for path in sorted(outputs_dir.rglob("*"))[-20:]:
        print(path)
else:
    print("Hydra outputs directory was not found.")

print("\nLatest checkpoints:")
for path in sorted(CHECKPOINT_DIR.rglob("*"))[-20:]:
    print(path)
